# NIMH Data Archive Report

A list of publications funded by the NIMH were downloaded from the NIH RePORTER website for years 2022-2025. The exports are stored in the `data/` directory.

## RePORTER Publication Data

Publications with a PMID listed were downloaded and processed using Oddpub.

In [1]:
from pub_data_parser import get_all_pmids

total_pmids: int = len(get_all_pmids())
print(f"Total PMIDs: {total_pmids}")


Total PMIDs: 16773


## Metapub Downloads

[Metapub](https://github.com/metapub/metapub) allows for the legal download publisher and Health and Human Services (HHS) manuscripts of of publications. If a publication url is not found, the Metapub library will provide a possible reason.

In [2]:
import pandas as pd

urls: pd.DataFrame = pd.read_csv("data/article_urls.csv")
urls.head()


,pmid,url,backup_url,reason,title
0,30924103,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://link.springer.com/article/10.1007/s109...,NaN,Does Religiosity Predict Blood Donation in Bra...
1,31454265,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://www.tandfonline.com/doi/full/10.1080/1...,NaN,Development of Overgeneral Autobiographical Me...
2,31868380,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://psycnet.apa.org:443/doiLanding?doi=10....,NaN,"Emotion expressivity, suicidal ideation, and e..."
3,32014421,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://linkinghub.elsevier.com/retrieve/pii/S...,NaN,Evaluation of pulmonary tuberculosis diagnosti...
4,32100255,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://link.springer.com/article/10.1007/s405...,NaN,Psychiatry Match Rates Increase After Exposure...


# Open Access URLs Provided



In [3]:
import plotly.express as px

def categorize_reason(reason: str):
    if reason == '':
        return "Reason: Found"
    elif reason.startswith('NOFORMAT'):
        return 'Metapub Cannot Parse Format'
    elif reason.startswith("PAYWALL"):
        return 'Article Paywalled'
    elif reason.startswith("TXERROR"):
        return 'Transaction Error'
    elif reason.startswith("DENIED"):
        return 'Url Access Denied'
    elif reason.startswith("Pubmed"):
        return 'PMID Not Found'
    else:
        return 'Other Error'


urls['reason'] = urls['reason'].fillna('')
urls['found_status'] = urls['reason'].map(lambda x: 'Open Access URL Found' if x == '' else 'Not Found')
urls['not_found_category'] = urls['reason'].map(lambda x: categorize_reason(x))
url_map = px.treemap(urls, path=['found_status'], values=urls['reason'].map(lambda x: 1),
           title="Proportion of Articles by URL Status",
           branchvalues="total"
           )
url_map.update_traces(textinfo="label+percent parent")


In [4]:
not_found_map = px.treemap(urls.loc[urls["found_status"] == "Not Found"], path=[px.Constant("Not Found"), "not_found_category"], values=urls.loc[urls["found_status"] == "Not Found"]['reason'].map(lambda x: 1),
           title="Not Found by Reason",
           branchvalues="total"
           )
not_found_map.update_traces(textinfo="label+percent parent")


## Article Downloads

The article URLs were automatically downloaded using Python GET requests. Failed downloads were listed as "Invalid Files" and successful downloads were listed as "Valid Files." 

Manucscipts could be either Heatlh and Human Services manuscirpts or publisher manuscripts.

In [5]:
from pathlib import Path
pd.options.mode.chained_assignment = None

file_inventory = Path("data/pdf_file_inventories_post_sort/")
inventory_files = list(file_inventory.glob("*.txt"))
pdf_types = []
for inventory_file in inventory_files:
    with open(inventory_file, 'r') as f_in:
        for line in f_in:
            pdf_types.append({
                "pmid": int(Path(line).stem),
                "file_type": inventory_file.stem.split("_")[0]
            })
pdfs = pd.DataFrame(pdf_types)
inventory_df = urls.merge(pdfs, on='pmid', how="left")
found_df = inventory_df.loc[inventory_df["found_status"] == "Open Access URL Found"]
found_df["file_type"] = found_df["file_type"].fillna("unknown")
found_df["Valid PDF"] = found_df["file_type"].map(lambda x: "Valid File" if x in ["publisher", "hhs"] else "Invalid File")
found_df["count"] = 1

inventory_map = px.treemap(found_df,
                           path=["Valid PDF", "file_type"], 
                           values="count",
           title="PDF Manuscripts by Type"
           )
inventory_map.update_traces(textinfo="label+percent parent")


## Oddpub Analysis of PDF Files

Oddpub was used to scrape the code and data availibity of the valid PDF files.

In [6]:
oddpub_df = pd.read_csv(r"./data/total_oddpub_results.csv")
oddpub_df["pmid"] = oddpub_df['article'].map(lambda x: int(x.split(".")[0]))
oddpub_df.head()


,X,article,is_open_data,open_data_category,is_open_code,open_data_statements,open_code_statements,source_dir,pmid
0,1,30924103.txt,False,NaN,False,NaN,NaN,sub_30,30924103
1,1,31454265.txt,False,NaN,False,NaN,NaN,sub_31,31454265
2,2,31868380.txt,False,NaN,False,NaN,NaN,sub_31,31868380
3,1,32014421.txt,False,NaN,False,NaN,NaN,sub_32,32014421
4,2,32100255.txt,False,NaN,False,NaN,NaN,sub_32,32100255


## Create merged dataframe with all information

In [7]:
total_inventory = inventory_df.merge(oddpub_df.drop(columns=["X"]), on="pmid", how="left")
reporter_files = Path(r"./data/").glob("*NIMH_pubs.csv")
reporter_dfs = []
for reporter_file in reporter_files:
    df = pd.read_csv(reporter_file)
    reporter_dfs.append(df)
reporter_df = pd.concat(reporter_dfs)

total_inventory.head()


,pmid,url,backup_url,reason,title,found_status,not_found_category,file_type,article,is_open_data,open_data_category,is_open_code,open_data_statements,open_code_statements,source_dir
0,30924103,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://link.springer.com/article/10.1007/s109...,,Does Religiosity Predict Blood Donation in Bra...,Open Access URL Found,Reason: Found,hhs,30924103.txt,False,NaN,False,NaN,NaN,sub_30
1,31454265,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://www.tandfonline.com/doi/full/10.1080/1...,,Development of Overgeneral Autobiographical Me...,Open Access URL Found,Reason: Found,hhs,31454265.txt,False,NaN,False,NaN,NaN,sub_31
2,31868380,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://psycnet.apa.org:443/doiLanding?doi=10....,,"Emotion expressivity, suicidal ideation, and e...",Open Access URL Found,Reason: Found,hhs,31868380.txt,False,NaN,False,NaN,NaN,sub_31
3,32014421,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://linkinghub.elsevier.com/retrieve/pii/S...,,Evaluation of pulmonary tuberculosis diagnosti...,Open Access URL Found,Reason: Found,hhs,32014421.txt,False,NaN,False,NaN,NaN,sub_32
4,32100255,http://europepmc.org/backend/ptpmcrender.fcgi?...,https://link.springer.com/article/10.1007/s405...,,Psychiatry Match Rates Increase After Exposure...,Open Access URL Found,Reason: Found,hhs,32100255.txt,False,NaN,False,NaN,NaN,sub_32


In [8]:
merged = total_inventory.merge(reporter_df, left_on="pmid", right_on='PMID', how="left")
# there are multple rows per PMID in the reporter_df due to multiple grants per publication
merged = merged.drop_duplicates(subset=["pmid"])
merged.to_csv(r"./data/nda_merged_inventory.csv", index=False)


## Percentage of publications funded by NIMH by publication year

In [27]:
# Total Number papers
import plotly.express as px

# Filter out 2025 and count papers by year
year_counts = merged[merged['Pub Year'] != 2025]['Pub Year'].value_counts().sort_index()

# Create bar plot using plotly express
fig = px.bar(
    x=year_counts.index, 
    y=year_counts.values,
    title='Number of Publications by Year',
    labels={'x': 'Publication Year', 'y': 'Number of Publications'},
    height=600
)

# Add value labels on top of bars
fig.update_traces(texttemplate='%{y}', textposition='outside')
fig.update_xaxes(dtick=1)
# Show the plot
fig.show()


In [26]:
# Filter for open data papers and exclude 2025
open_data_by_year = merged[
    (merged['is_open_data'] == True) & 
    (merged['Pub Year'] != 2025)
]['Pub Year'].value_counts().sort_index()

# Create bar plot
fig = px.bar(
    x=open_data_by_year.index,
    y=open_data_by_year.values, 
    title='Number of Publications with Open Data by Year',
    labels={'x': 'Publication Year', 'y': 'Number of Publications'},
    height=600
)

# Add value labels on top of bars
fig.update_traces(texttemplate='%{y}', textposition='outside')
fig.update_xaxes(dtick=1)

# Show the plot
fig.show()


In [11]:
# Define NDA-related patterns to search for
nda_patterns = [
    r'\bNDA\b',  # Standalone NDA
    r'NIMH\s+Data\s+Archive',  # NIMH Data Archive with flexible spacing
    r'National\s+Institute\s+(?:of\s+)?Mental\s+Health\s+Data\s+Archive',  # Full name with optional 'of'
    r'National\s+Database\s+for\s+Autism\s+Research',  # NDAR full name
    r'\bNDAR\b',  # Standalone NDAR
    r'Mental\s+Health\s+Archive',
    r'https://nda.nih.gov/',
    r'nda.nih.gov',
    r'nda.nih'  # Shortened version
    r'ndar.nih.gov',
    r'ndar.nih'
]

# Combine patterns into single regex with OR operator
combined_pattern = '|'.join(nda_patterns)

# Filter for rows that have NDA-related terms in open_data_statements
nda_mentions = merged[
    merged['open_data_statements'].str.contains(
        combined_pattern, 
        case=False, 
        regex=True, 
        na=False
    )
]

nda_mentions["open_data_statements"].to_csv(r"./data/nda_mentions.csv", index=False)


In [22]:
# Filter for open data papers and exclude 2025
nda_by_year = nda_mentions[
    (nda_mentions['is_open_data'] == True) & 
    (nda_mentions['Pub Year'] != 2025)
]['Pub Year'].value_counts().sort_index()

# Create bar plot
fig = px.bar(
    x=nda_by_year.index,
    y=nda_by_year.values,
    title='Number of Publications with Open Data by Year', 
    labels={'x': 'Publication Year', 'y': 'Number of Publications'},
    height=600,  # Increase height to prevent label cutoff
)

# Add value labels on top of bars
fig.update_traces(texttemplate='%{y}', textposition='outside')
fig.update_xaxes(dtick=1)

# Show the plot
fig.show()
